# PyTorch Compile Analysis

In [ ]:

import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt


## Part 1: Backend Comparison

In [ ]:

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512,512)
        self.fc2 = nn.Linear(512,512)
        self.fc3 = nn.Linear(512,512)
        self.relu = nn.ReLU()

    def forward(self,x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Net().to(device)

x = torch.randn(512,512).to(device)
y = torch.randn(512,512).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters())


In [ ]:

backends = ["eager","aot_eager","inductor","cudagraphs"]
times = []

for backend in backends:

    if backend == "eager":
        compiled = model
    else:
        compiled = torch.compile(model, backend=backend)

    start = time.time()

    for _ in range(100):
        optimizer.zero_grad()
        out = compiled(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()

    times.append((time.time() - start)/100)

print(times)


In [ ]:

plt.figure()
plt.bar(backends, times)
plt.ylabel("Average time per iteration")
plt.title("Backend Performance Comparison")
plt.show()


## Part 2: Debugging Compilation Failures

In [ ]:

def problem1(x):
    if x.sum() > 0:
        return x * 2
    else:
        return x / 2

def problem2(x):
    d = {}
    d['key'] = x
    return d['key'] * 2

def problem3(x):
    result = 0
    for i in range(10):
        result += (x ** i).sum()
    return result


In [ ]:

compiled1 = torch.compile(problem1)
compiled2 = torch.compile(problem2)
compiled3 = torch.compile(problem3)



Fixes generally involve removing Python-side control flow and replacing it with tensor operations such as `torch.where`, and avoiding Python containers that Dynamo cannot trace.


## Part 3: Graph Capture

In [ ]:

def graph_function(x,w1,w2):

    y = x @ w1
    y = torch.relu(y)
    y = y @ w2

    y = y + x

    y = torch.nn.functional.layer_norm(y,y.shape)

    print("Hello")

    lst = []
    lst.append(1)

    return y


In [ ]:

compiled = torch.compile(graph_function, fullgraph=False)
